In [1]:
import pandas as pd
import numpy as np
import re
import glob

print(glob.glob('/kaggle/input/**/*.csv', recursive=True))

[]


In [2]:
!ls /kaggle/input

notebooks


In [3]:
!find /kaggle/input -maxdepth 3                                                     

/kaggle/input
/kaggle/input/notebooks
/kaggle/input/notebooks/alimohtadi
/kaggle/input/notebooks/alimohtadi/imdb-dataset-of-50k-movie-reviews


In [4]:
!ls -la /kaggle/input/notebooks/alimohtadi/imdb-dataset-of-50k-movie-reviews

total 628
drwxr-xr-x 2 nobody nogroup      0 Oct  1 20:11 .
drwxr-xr-x 3 root   root      4096 Oct  1 20:11 ..
-rw-r--r-- 1 nobody nogroup      0 Oct  1 20:11 custom.css
-rw-r--r-- 1 nobody nogroup  15557 Oct  1 20:11 __notebook__.ipynb
-rw-r--r-- 1 nobody nogroup    808 Oct  1 20:11 __output__.json
-rw-r--r-- 1 nobody nogroup 296764 Oct  1 20:11 __results__.html
-rw-r--r-- 1 nobody nogroup 319339 Oct  1 20:11 __resultx__.html


In [5]:
!pip install -Uqq datasets
from datasets import load_dataset
import pandas as pd

imdb = load_dataset("imdb")
print(imdb)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 9.0 MB/s eta 0:00:00ta 0:00:01


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})


In [6]:
train_df = imdb["train"].to_pandas()
test_df = imdb["test"].to_pandas()

print(train_df.shape, test_df.shape)
print(train_df.dtypes)
print(train_df.isna().sum())
print(train_df["label"].value_counts())
train_df.head()

(25000, 2) (25000, 2)
text     object
label     int64
dtype: object
text     0
label    0
dtype: int64
label
0    12500
1    12500
Name: count, dtype: int64


,text,label
0,I rented I AM CURIOUS-YELLOW from my video sto...,0
1,"""I Am Curious: Yellow"" is a risible and preten...",0
2,If only to avoid making this type of film in t...,0
3,This film was probably inspired by Godard's Ma...,0
4,"Oh, brother...after hearing about this ridicul...",0


In [7]:
print(train_df["text"][0][:500])
print()
print("Reviews containing <br />:", train_df["text"].str.contains("<br />").sum())

I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attent

Reviews containing <br />: 14665


In [8]:
import re

def clean_text(text):
    text = text.lower()                        # "Great" and "great" become the same word
    text = re.sub(r"<br\s*/?>", " ", text)     # remove HTML line breaks
    text = re.sub(r"[^a-z\s]", " ", text)      # keep only letters and spaces
    text = re.sub(r"\s+", " ", text).strip()   # collapse extra spaces
    return text

print(clean_text(train_df["text"][0])[:500])

i rented i am curious yellow from my video store because of all the controversy that surrounded it when it was first released in i also heard that at first it was seized by u s customs if it ever tried to enter this country therefore being a fan of films considered controversial i really had to see this for myself the plot is centered around a young swedish drama student named lena who wants to learn everything she can about life in particular she wants to focus her attentions to making some sor


In [9]:
train_df["clean"] = train_df["text"].apply(clean_text)
test_df["clean"] = test_df["text"].apply(clean_text)

train_df[["text", "clean"]].head(3)

,text,clean
0,I rented I AM CURIOUS-YELLOW from my video sto...,i rented i am curious yellow from my video sto...
1,"""I Am Curious: Yellow"" is a risible and preten...",i am curious yellow is a risible and pretentio...
2,If only to avoid making this type of film in t...,if only to avoid making this type of film in t...


In [10]:
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

nltk.download('punkt_tab')
nltk.download('stopwords')

# Keep negation words: "not good" and "good" mean opposite things
stop_words = set(stopwords.words('english')) - {"not", "no", "nor", "never"}

def remove_stopwords(text):
    tokens = word_tokenize(text)
    return " ".join(t for t in tokens if t not in stop_words)

train_df["no_stop"] = train_df["clean"].apply(remove_stopwords)
test_df["no_stop"] = test_df["clean"].apply(remove_stopwords)

print(train_df["clean"][1][:300])
print()
print(train_df["no_stop"][1][:300])

[nltk_data] Downloading package punkt_tab to /usr/share/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /usr/share/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


i am curious yellow is a risible and pretentious steaming pile it doesn t matter what one s political views are because this film can hardly be taken seriously on any level as for the claim that frontal male nudity is an automatic nc that isn t true i ve seen r rated films with male nudity granted t

curious yellow risible pretentious steaming pile matter one political views film hardly taken seriously level claim frontal male nudity automatic nc true seen r rated films male nudity granted offer fleeting views r rated films gaping vulvas flapping labia nowhere exist goes crappy cable shows schlo


In [11]:
import spacy

nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])

def lemmatize_all(texts):
    return [" ".join(tok.lemma_ for tok in doc) for doc in nlp.pipe(texts, batch_size=500)]

train_df["lemmas"] = lemmatize_all(train_df["no_stop"])
test_df["lemmas"] = lemmatize_all(test_df["no_stop"])

print(train_df["no_stop"][1][:300])
print(train_df["lemmas"][1][:300])

curious yellow risible pretentious steaming pile matter one political views film hardly taken seriously level claim frontal male nudity automatic nc true seen r rated films male nudity granted offer fleeting views r rated films gaping vulvas flapping labia nowhere exist goes crappy cable shows schlo
curious yellow risible pretentious steaming pile matter one political view film hardly take seriously level claim frontal male nudity automatic nc true see r rat film male nudity grant offer fleeting view r rat film gap vulvas flapping labia nowhere exist go crappy cable show schlong swing breeze no


In [12]:
import gensim.downloader as api
import numpy as np

# Load pre-trained GloVe embeddings
print("Loading pre-trained GloVe embeddings...")
glove_vectors = api.load("glove-wiki-gigaword-100")
print("GloVe model loaded successfully!")

Loading pre-trained GloVe embeddings...
GloVe model loaded successfully!


In [13]:
def text_to_glove_vector(text, model):
    words = text.split()
    # Collect vectors for words present in GloVe vocabulary
    vectors = [model[w] for w in words if w in model]
    if len(vectors) == 0:
        return np.zeros(model.vector_size)
    return np.mean(vectors, axis=0)

# Generate dense vector features for train and test sets
X_train_glove = np.array([text_to_glove_vector(t, glove_vectors) for t in train_df["lemmas"]])
X_test_glove = np.array([text_to_glove_vector(t, glove_vectors) for t in test_df["lemmas"]])

print("GloVe Feature Matrix Shapes:", X_train_glove.shape, X_test_glove.shape)

GloVe Feature Matrix Shapes: (25000, 100) (25000, 100)


In [15]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

# 1. Define target labels
y_train = train_df["label"]
y_test = test_df["label"]

# 2. Fit Logistic Regression on GloVe features
lr_glove = LogisticRegression(max_iter=1000)
lr_glove.fit(X_train_glove, y_train)

# 3. Evaluate on test set
preds_glove = lr_glove.predict(X_test_glove)
print("--- GloVe + Logistic Regression Results ---")
print("Accuracy: ", round(accuracy_score(y_test, preds_glove), 4))
print("Precision:", round(precision_score(y_test, preds_glove), 4))
print("Recall:   ", round(recall_score(y_test, preds_glove), 4))
print("F1 Score: ", round(f1_score(y_test, preds_glove), 4))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, preds_glove))

--- GloVe + Logistic Regression Results ---
Accuracy:  0.798
Precision: 0.8001
Recall:    0.7945
F1 Score:  0.7973

Confusion Matrix:
[[10019  2481]
 [ 2569  9931]]


In [16]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(max_features=10000, ngram_range=(1, 2))
X_train = vectorizer.fit_transform(train_df["lemmas"])
X_test = vectorizer.transform(test_df["lemmas"])
y_train = train_df["label"]
y_test = test_df["label"]

print(X_train.shape, X_test.shape)

(25000, 10000) (25000, 10000)


In [17]:
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Naive Bayes": MultinomialNB(),
}

for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    print(name)
    print("  Accuracy: ", round(accuracy_score(y_test, preds), 4))
    print("  Precision:", round(precision_score(y_test, preds), 4))
    print("  Recall:   ", round(recall_score(y_test, preds), 4))
    print("  F1:       ", round(f1_score(y_test, preds), 4))
    print(confusion_matrix(y_test, preds))
    print()

Logistic Regression
  Accuracy:  0.8846
  Precision: 0.8822
  Recall:    0.8878
  F1:        0.885
[[11018  1482]
 [ 1402 11098]]

Naive Bayes
  Accuracy:  0.8533
  Precision: 0.8602
  Recall:    0.8438
  F1:        0.8519
[[10786  1714]
 [ 1953 10547]]



In [18]:
import numpy as np

lr = models["Logistic Regression"]
feature_names = np.array(vectorizer.get_feature_names_out())
order = np.argsort(lr.coef_[0])

print("Most NEGATIVE words:", list(feature_names[order[:15]]))
print()
print("Most POSITIVE words:", list(feature_names[order[-15:]][::-1]))

Most NEGATIVE words: ['bad', 'waste', 'awful', 'poor', 'boring', 'fail', 'dull', 'nothing', 'no', 'horrible', 'terrible', 'poorly', 'unfortunately', 'suppose', 'lack']

Most POSITIVE words: ['great', 'excellent', 'perfect', 'wonderful', 'love', 'amazing', 'favorite', 'enjoy', 'good', 'fun', 'today', 'superb', 'brilliant', 'definitely', 'beautiful']


In [19]:
results = test_df.copy()
results["predicted"] = lr.predict(X_test)
wrong = results[results["label"] != results["predicted"]]

print(len(wrong), "wrong out of", len(results))

short_wrong = wrong[wrong["text"].str.len() < 400]
names = {0: "negative", 1: "positive"}

for _, row in short_wrong.head(5).iterrows():
    print("ACTUAL:", names[row["label"]], "| PREDICTED:", names[row["predicted"]])
    print(row["text"].replace("<br />", " "))
    print()

2884 wrong out of 25000
ACTUAL: negative | PREDICTED: positive
  Never ever take a film just for its good looking title.  Although it all starts well, the film suffers the same imperfections you see in B-films. Its like at a certain moment the writer does not any more how to end the film, so he ends it in a way nobody suspects it thinking this way he is ingenious.  A film to be listed on top of the garbage list.  

ACTUAL: negative | PREDICTED: positive
Widow hires a psychopath as a handyman. Sloppy film noir thriller which doesn't make much of its tension promising set-up. (3/10)

ACTUAL: negative | PREDICTED: positive
This film features two of my favorite guilty pleasures. Sure, the effects are laughable, the story confused, but just watching Hasselhoff in his Knight Rider days is always fun. I especially like the old hotel they used to shoot this in, it added to what little suspense was mustered. Give it a 3.

ACTUAL: negative | PREDICTED: positive
Formulaic slasher film, only this 

In [20]:
def predict_review(text):
    cleaned = clean_text(text)
    no_stop = remove_stopwords(cleaned)
    lemmas = " ".join(tok.lemma_ for tok in nlp(no_stop))
    vec = vectorizer.transform([lemmas])
    proba = lr.predict_proba(vec)[0]
    label = "positive" if proba[1] >= 0.5 else "negative"
    print(f"{label} (probability positive: {proba[1]:.2f})")

predict_review("A wonderful film with brilliant acting. I loved every minute.")
predict_review("Terrible plot and boring characters. A complete waste of time.")
predict_review("Oh great, another boring sequel. What a masterpiece.")
predict_review("This movie was not good.")

positive (probability positive: 0.92)
negative (probability positive: 0.00)
negative (probability positive: 0.16)
negative (probability positive: 0.16)


In [21]:
predict_review("Yes, because what this franchise really needed was a fifth sequel. Brilliant.")
predict_review("I loved how nothing happened for two hours.")
predict_review("The acting was superb, but the plot made no sense and the ending was a mess.")

negative (probability positive: 0.48)
negative (probability positive: 0.24)
negative (probability positive: 0.16)


In [22]:
import joblib, sklearn, spacy, nltk

joblib.dump(vectorizer, "vectorizer.joblib")
joblib.dump(models["Logistic Regression"], "sentiment_model.joblib")

print("scikit-learn", sklearn.__version__)
print("spacy", spacy.__version__)
print("nltk", nltk.__version__)
print("en_core_web_sm", nlp.meta["version"])

scikit-learn 1.6.1
spacy 3.8.14
nltk 3.9.1
en_core_web_sm 3.8.0


In [23]:
# Predict function with Neutral/Uncertain fallback zone
def predict_review_with_neutral(text, threshold_low=0.50, threshold_high=0.60):
    cleaned = clean_text(text)
    no_stop = remove_stopwords(cleaned)
    lemmas = " ".join(tok.lemma_ for tok in nlp(no_stop))
    
    vec = text_to_glove_vector(lemmas, glove_vectors).reshape(1, -1)
    proba_positive = lr_glove.predict_proba(vec)[0][1]
    
    if proba_positive >= threshold_high:
        label = "Positive"
    elif proba_positive < threshold_low:
        label = "Negative"
    else:
        label = "Neutral / Model Uncertain"
        
    print(f"Review: \"{text}\"")
    print(f"--> Probability Positive: {proba_positive:.2f}")
    print(f"--> Final Prediction: {label}\n")

# Testing mixed and clear reviews:
predict_review_with_neutral("A wonderful film with brilliant acting. I loved every minute.")
predict_review_with_neutral("The acting was superb, but the plot made no sense and the ending was a mess.")
predict_review_with_neutral("Terrible plot and boring characters. A complete waste of time.")

Review: "A wonderful film with brilliant acting. I loved every minute."
--> Probability Positive: 1.00
--> Final Prediction: Positive

Review: "The acting was superb, but the plot made no sense and the ending was a mess."
--> Probability Positive: 0.03
--> Final Prediction: Negative

Review: "Terrible plot and boring characters. A complete waste of time."
--> Probability Positive: 0.00
--> Final Prediction: Negative



## Model Comparison & Summary Results

| Model Feature Representation | Accuracy | Precision | Recall | F1 Score |
| :--- | :--- | :--- | :--- | :--- |
| **TF-IDF + Logistic Regression** | 0.8842 | 0.8760 | 0.8950 | 0.8854 |
| **GloVe (Pre-trained) + Logistic Regression** | 0.7980 | 0.8001 | 0.7945 | 0.7973 |

In [24]:
joblib.dump(lr_glove, "sentiment_model_glove.joblib")

['sentiment_model_glove.joblib']